In [1]:
import pandas as pd
import numpy as np

In [2]:
renewal_calls = pd.read_csv('../../data/raw/renewal_calls.csv', low_memory=False)
renewal_calls.columns = renewal_calls.columns.str.lower().str.replace(' ', '_')
renewal_calls.shape

(186534, 41)

Removing duplicates

In [3]:
renewal_calls.duplicated().sum()

np.int64(28812)

In [4]:
renewal_calls = renewal_calls.drop_duplicates()
renewal_calls.shape

(157722, 41)

Dropping rows where co_ref is null because it acts as primary key to join with other datasets

In [5]:
renewal_calls['co_ref'].isnull().sum()

np.int64(4453)

In [6]:
renewal_calls = renewal_calls.dropna(subset=['co_ref'])
renewal_calls.shape

(153269, 41)

Standardizing call_direction column values

In [7]:
renewal_calls['call_direction'].value_counts()

call_direction
OUT_BOUND    65852
Outbound     65616
IN_BOUND     11149
Inbound      10652
Name: count, dtype: int64

In [8]:
renewal_calls['call_direction'] = renewal_calls['call_direction'].replace({
    'OUT_BOUND': 'Outbound',
    'IN_BOUND': 'Inbound'
})
renewal_calls['call_direction'].value_counts()

call_direction
Outbound    131468
Inbound      21801
Name: count, dtype: int64

Converting proper datatype of dates

In [9]:
renewal_calls['call_date'].dtype

<StringDtype(storage='python', na_value=nan)>

In [10]:
renewal_calls['call_date'] = pd.to_datetime(renewal_calls['call_date'], format='%d-%m-%Y')
renewal_calls['call_date'].dtype

dtype('<M8[us]')

In [11]:
renewal_calls['call_date'].head(5)

0   2025-01-29
1   2025-02-26
2   2025-01-24
3   2025-06-09
4   2024-08-20
Name: call_date, dtype: datetime64[us]

Dropping columns with all nulls

In [12]:
empty_columns = renewal_calls.columns[renewal_calls.isnull().all()]
renewal_calls = renewal_calls.drop(columns=empty_columns)
renewal_calls.shape

(153269, 40)

Dropping analysed_call as it has only 1 unique value

In [13]:
renewal_calls['analysed_call'].nunique()

1

In [14]:
renewal_calls = renewal_calls.drop('analysed_call', axis=1)

Dropping columns with more than 90% null data

In [15]:
drop_columns = [
    'argument_that_convinced_customer_to_stay_category',
    'agent_response_to_cancel_category',
    'justification_category',
    'reason_for_renewal_category',
]
for col in drop_columns:
    print(f"Null percentage of {col}: ", (renewal_calls[col].isnull().sum()/len(renewal_calls))*100)

Null percentage of argument_that_convinced_customer_to_stay_category:  98.86147883786023
Null percentage of agent_response_to_cancel_category:  96.86564145391436
Null percentage of justification_category:  91.7922084700755
Null percentage of reason_for_renewal_category:  89.90337250194104


In [16]:
renewal_calls = renewal_calls.drop(columns=drop_columns)
renewal_calls.shape

(153269, 35)

Dropping irrelevant columns<br>
call_year -> can be derived from call_date

In [17]:
renewal_calls['call_year'].unique()

array([2025, 2024, 2026])

In [18]:
renewal_calls = renewal_calls.drop('call_year', axis=1)
renewal_calls.shape

(153269, 34)

Saving cleaned dataset

In [19]:
import os
os.makedirs('../../data/cleaned', exist_ok=True)
renewal_calls.to_csv('../../data/cleaned/renewal_calls_cleaned.csv', index=False)
print('Saved! Shape:', renewal_calls.shape)

Saved! Shape: (153269, 34)
